# AutoEIT Test I — Transcription
## GSoC 2026 — Tanishq Jain

## Step 1: Install

In [1]:
!pip install openai-whisper openpyxl -q
!apt-get install -y ffmpeg -q 2>/dev/null
import whisper, os, re, json, subprocess, tempfile, unicodedata, openpyxl
from google.colab import files
print("Ready | Whisper:", whisper.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 41.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
Ready | Whisper: 20250625


## Step 2: Upload Files
Upload the 4 MP3 files + `AutoEIT_Sample_Audio_for_Transcribing.xlsx`

In [2]:
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(fname, 'wb') as f: f.write(data)
    print(fname, round(len(data)/1024/1024,1), 'MB')

Saving 038010_EIT-2A.mp3 to 038010_EIT-2A.mp3
Saving 038011_EIT-1A.mp3 to 038011_EIT-1A.mp3
Saving 038012_EIT-2A.mp3 to 038012_EIT-2A.mp3
Saving 038015_EIT-1A.mp3 to 038015_EIT-1A.mp3
Saving AutoEIT Sample Audio for Transcribing.xlsx to AutoEIT Sample Audio for Transcribing.xlsx
038010_EIT-2A.mp3 8.9 MB
038011_EIT-1A.mp3 9.1 MB
038012_EIT-2A.mp3 18.2 MB
038015_EIT-1A.mp3 8.6 MB
AutoEIT Sample Audio for Transcribing.xlsx 0.1 MB


## Step 3: Load Whisper
*Runtime → Change runtime type → T4 GPU first!*

In [3]:
print("Loading Whisper large...")
model = whisper.load_model("large")
print("Ready!")

Loading Whisper large...


100%|█████████████████████████████████████| 2.88G/2.88G [00:32<00:00, 94.8MiB/s]


Ready!


## Step 4: Stimuli & Participants

In [4]:
STIMULI = [
    "Quiero cortarme el pelo",
    "El libro está en la mesa",
    "El carro lo tiene Pedro",
    "Él se ducha cada mañana",
    "¿Qué dice usted que va a hacer hoy?",
    "Dudo que sepa manejar muy bien",
    "Las calles de esta ciudad son muy anchas",
    "Puede que llueva mañana todo el día",
    "Las casas son muy bonitas pero caras",
    "Me gustan las películas que acaban bien",
    "El chico con el que yo salgo es español",
    "Después de cenar me fui a dormir tranquilo",
    "Quiero una casa en la que vivan mis animales",
    "A nosotros nos fascinan las fiestas grandiosas",
    "Ella sólo bebe cerveza y no come nada",
    "Me gustaría que el precio de las casas bajara",
    "Cruza a la derecha y después sigue todo recto",
    "Ella ha terminado de pintar su apartamento",
    "Me gustaría que empezara a hacer más calor pronto",
    "El niño al que se le murió el gato está triste",
    "Una amiga mía cuida a los niños de mi vecino",
    "El gato que era negro fue perseguido por el perro",
    "Antes de poder salir él tiene que limpiar su cuarto",
    "La cantidad de personas que fuman ha disminuido",
    "Después de llegar a casa del trabajo tomé la cena",
    "El ladrón al que atrapó la policía era famoso",
    "Le pedí a un amigo que me ayudara con la tarea",
    "El examen no fue tan difícil como me habían dicho",
    "¿Serías tan amable de darme el libro que está en la mesa?",
    "Hay mucha gente que no toma nada para el desayuno",
]

PARTICIPANTS = [
    {"id": "038010", "file": "038010_EIT-2A.mp3"},
    {"id": "038011", "file": "038011_EIT-1A.mp3"},
    {"id": "038012", "file": "038012_EIT-2A.mp3"},
    {"id": "038015", "file": "038015_EIT-1A.mp3"},
]
print(len(STIMULI), "stimuli |", len(PARTICIPANTS), "participants")

30 stimuli | 4 participants


## Step 5: Matching Function

In [5]:
import unicodedata
import re
import difflib
import subprocess

def norm(t):
    t = t.lower().strip()
    t = unicodedata.normalize("NFD", t)
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"[^a-z0-9 ]", "", t)
    return re.sub(r" +", " ", t).strip()

def text_similarity(text1, text2):
    return difflib.SequenceMatcher(None, norm(text1), norm(text2)).ratio()

def assign_to_stimuli(all_segments, stimuli):
    segs = [s for s in all_segments if len(s["text"].strip()) > 2]

    if not segs:
        return ["[inaudible]"] * len(stimuli)

    segs = sorted(segs, key=lambda x: x["start"])
    n = len(segs)
    m = len(stimuli)

    dp = [[0.0] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sim = text_similarity(segs[i-1]["text"], stimuli[j-1])
            dp[i][j] = max(
                dp[i-1][j],
                dp[i][j-1],
                dp[i-1][j-1] + sim
            )

    result = ["[inaudible]"] * m
    i, j = n, m
    matches = []

    while i > 0 and j > 0:
        if dp[i][j] == dp[i-1][j]:
            i -= 1
        elif dp[i][j] == dp[i][j-1]:
            j -= 1
        else:
            sim = text_similarity(segs[i-1]["text"], stimuli[j-1])
            if sim > 0.15:
                matches.append((j-1, segs[i-1]["text"].strip()))
            i -= 1
            j -= 1

    for stim_idx, text in matches:
        result[stim_idx] = text

    return result

def extract_audio(inp, start_sec, out, duration=300):
    cmd = [
        "ffmpeg", "-y",
        "-ss", str(start_sec),
        "-i", inp,
        "-t", str(duration),
        "-acodec", "pcm_s16le",
        "-ar", "16000",
        "-ac", "1",
        out
    ]
    return subprocess.run(cmd, capture_output=True).returncode == 0

## Step 6: Transcribe All Participants
*~10-15 min each — do NOT close tab*

In [6]:
all_results = {}

for p in PARTICIPANTS:
    pid, afile = p["id"], p["file"]
    print(f"\n{'='*60}")
    print(f"Participant {pid} | {afile}")

    if not os.path.exists(afile):
        print("FILE NOT FOUND")
        all_results[pid] = ["[file not found]"] * 30
        continue

    # Convert to WAV for Whisper (full file, no skipping)
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False); tmp.close()
    cmd = ["ffmpeg","-y","-i",afile,"-acodec","pcm_s16le","-ar","16000","-ac","1",tmp.name]
    r = subprocess.run(cmd, capture_output=True)
    if r.returncode != 0:
        print("CONVERSION FAILED"); all_results[pid] = ["[error]"]*30; continue

    print("  Transcribing full audio with Whisper large...")
    result = model.transcribe(
        tmp.name,
        language="es",
        task="transcribe",
        word_timestamps=True,
        verbose=False,
        condition_on_previous_text=False,
        temperature=0.0,
        no_speech_threshold=0.3,
        compression_ratio_threshold=2.4,
    )
    os.unlink(tmp.name)

    segs = result.get("segments", [])
    print(f"  Total segments: {len(segs)}")
    print(f"  Full text preview: {result.get('text','')[:400]}")

    transcriptions = assign_to_stimuli(segs, STIMULI)
    all_results[pid] = transcriptions

    filled = sum(1 for t in transcriptions if "[inaudible]" not in t)
    print(f"  Matched: {filled}/30")
    for i in range(min(5, 30)):
        print(f"    {i+1:2d}. {STIMULI[i][:35]:35s} | {transcriptions[i][:50]}")

print("\nAll done!")


Participant 038010 | 038010_EIT-2A.mp3
  Transcribing full audio with Whisper large...


100%|██████████| 54963/54963 [00:54<00:00, 1011.46frames/s]


  Total segments: 55
  Full text preview:  Gracias. Gracias. We drove to the park. I'll call her tomorrow night. You can buy meat at the butcher shop. My brother just bought a brand new computer. Sometimes they take their dog for a walk in the park. We're going to play volleyball at the gym that I told you about. That was the last English sentence. Now, you're going to hear a number of sentences in Spanish. Once again, after each sentence
  Matched: 30/30
     1. Quiero cortarme el pelo             | Quiero cortarme el pelo.
     2. El libro está en la mesa            | El libro está en la mesa.
     3. El carro lo tiene Pedro             | El carro no tiene pelo.
     4. Él se ducha cada mañana             | Él se ducha cada mañana.
     5. ¿Qué dice usted que va a hacer hoy? | ¿Qué dice usted que va a hacer hoy?

Participant 038011 | 038011_EIT-1A.mp3
  Transcribing full audio with Whisper large...


100%|██████████| 54703/54703 [00:47<00:00, 1163.32frames/s]


  Total segments: 52
  Full text preview:  Ahora vamos a escuchar algunas frases en inglés. ¡Gracias! We drove to the park. I'll call her tomorrow night. You can buy meat at the butcher shop. My brother just bought a brand new computer. Sometimes they take their dog for a walk in the park. We're going to play volleyball at the gym that I told you about. ¿Qué pasa? Quiero cortarme el pelo. El libro está en la mesa. El carro no tiene pelo. 
  Matched: 30/30
     1. Quiero cortarme el pelo             | Quiero cortarme el pelo.
     2. El libro está en la mesa            | El libro está en la mesa.
     3. El carro lo tiene Pedro             | El carro no tiene pelo.
     4. Él se ducha cada mañana             | Él se ducha cada mañana.
     5. ¿Qué dice usted que va a hacer hoy? | ¿Qué dice usted que va a hacer hoy?

Participant 038012 | 038012_EIT-2A.mp3
  Transcribing full audio with Whisper large...


100%|██████████| 112554/112554 [01:38<00:00, 1142.95frames/s]


  Total segments: 117
  Full text preview:  Clicca en Play y puedes ir a hacer tu grabación. Al final vas a dar clic en Stop. ¿Ok? Comienza y termina. Gracias. ... ... ... ... ... ... ... ... ... ... ... ... Puedes comprar carne en la tienda de pescado. Mi hermano solo trajo un computador nuevo. A veces tocan a su perro por una caminata en el parque. Vamos a jugar vóleybol en el gimnasio de Tojo. Quiero convertirme en un hombre. El pez est
  Matched: 29/30
     1. Quiero cortarme el pelo             | Quiero cortar el pelo.
     2. El libro está en la mesa            | El libro está en la mesa.
     3. El carro lo tiene Pedro             | El carro no tiene pelo.
     4. Él se ducha cada mañana             | El se duche cada mañana.
     5. ¿Qué dice usted que va a hacer hoy? | ¿Qué dice usted que va a hacer hoy?

Participant 038015 | 038015_EIT-1A.mp3
  Transcribing full audio with Whisper large...


 94%|█████████▍| 49937/52937 [00:47<00:02, 1048.04frames/s]


  Total segments: 54
  Full text preview:  Gracias. y y y y y y y y y y y y y y Voy a llamarla mañana noche. Puedes comprar carne en la tienda de la maquina. Mi hermano solo compró un computador nuevo. A veces tocan a su perro en el parque. Vamos a jugar al vóleybol en el gimnasio que te he contado. Quiero cortarme el pelo. El libro está en la mesa. El carro no tiene pelo. Las casas son muy bonitas, pero caras. Él se ducha cada mañana. Me
  Matched: 29/30
     1. Quiero cortarme el pelo             | Quiero cortarme el pelo.
     2. El libro está en la mesa            | El libro está en la mesa.
     3. El carro lo tiene Pedro             | El carro no tiene pelo.
     4. Él se ducha cada mañana             | Él se ducha cada mañana.
     5. ¿Qué dice usted que va a hacer hoy? | ¿Qué dice usted que va a hacer hoy?

All done!


## Step 7: Review & Manually Fix Any Wrong Ones

In [7]:
import pandas as pd
rows = []
for i, s in enumerate(STIMULI):
    row = {"#": i+1, "Target": s[:38]}
    for pid in ["038010","038011","038012","038015"]:
        t = all_results.get(pid, [""]*30)[i]
        row[pid] = (t or "")[:45]
    rows.append(row)
pd.set_option("display.max_colwidth", 46)
display(pd.DataFrame(rows))

,#,Target,038010,038011,038012,038015
0,1,Quiero cortarme el pelo,Quiero cortarme el pelo.,Quiero cortarme el pelo.,Quiero cortar el pelo.,Quiero cortarme el pelo.
1,2,El libro está en la mesa,El libro está en la mesa.,El libro está en la mesa.,El libro está en la mesa.,El libro está en la mesa.
2,3,El carro lo tiene Pedro,El carro no tiene pelo.,El carro no tiene pelo.,El carro no tiene pelo.,El carro no tiene pelo.
3,4,Él se ducha cada mañana,Él se ducha cada mañana.,Él se ducha cada mañana.,El se duche cada mañana.,Él se ducha cada mañana.
4,5,¿Qué dice usted que va a hacer hoy?,¿Qué dice usted que va a hacer hoy?,¿Qué dice usted que va a hacer hoy?,¿Qué dice usted que va a hacer hoy?,¿Qué dice usted que va a hacer hoy?
5,6,Dudo que sepa manejar muy bien,Dudo que sepa manejar muy bien.,Todo que sepa manejar muy bien.,¿Todo se calesar también?,[inaudible]
6,7,Las calles de esta ciudad son muy anch,Las calles de esta ciudad son muy anchas.,Las calles de esta ciudad son muy anchas.,"La calle Sentee, los calles altos.",hoy. Tuvo que sepamanajar bien. Las calles de
7,8,Puede que llueva mañana todo el día,Puede que llueva mañana todo el día.,Puede que llueva mañana todo el día.,El pueblo se que ella calla.,Lleva mañana todo el día.
8,9,Las casas son muy bonitas pero caras,Las casas son muy bonitas pero caras.,Las casas son muy bonitas pero caras.,Las casas son bonitas pero caras.,Las casas son muy bonitos pero caras.
9,10,Me gustan las películas que acaban bie,Me gustan las películas que acaban bien.,Me gustan las películas que acaban bien.,Me gustan las películas de Pecan bien.,Me gusta las películas que acaban bien.


## Step 8: Manual Fixes

If any sentence is wrong, fix it here before saving. Example:
```python
all_results["038010"][0] = "Quiero cortarme el pelo"
```
(Index 0 = sentence 1, index 29 = sentence 30)

In [8]:
# ── ADD CORRECTIONS HERE IF NEEDED ──


print("Corrections done")
for pid in ["038010","038011","038012","038015"]:
    t = all_results.get(pid, [])
    ok = sum(1 for x in t if x not in ["[inaudible]","[file not found]","[error]",""])
    print(f"  {pid}: {ok}/30 filled")

Corrections done
  038010: 30/30 filled
  038011: 30/30 filled
  038012: 29/30 filled
  038015: 29/30 filled


## Step 9: Save to Excel

In [9]:
wb = openpyxl.load_workbook("AutoEIT Sample Audio for Transcribing.xlsx")
sheet_map = {"038010":"38010-2A","038011":"38011-1A","038012":"38012-2A","038015":"38015-1A"}
for pid, sname in sheet_map.items():
    if pid not in all_results or sname not in wb.sheetnames: continue
    ws = wb[sname]; trans = all_results[pid]; n = 0
    for row in ws.iter_rows():
        for cell in row:
            if cell.column==1 and isinstance(cell.value,(int,float)):
                idx = int(cell.value)
                if 1 <= idx <= 30:
                    ws.cell(row=cell.row, column=3, value=trans[idx-1]); n+=1
    print(f"  {pid} ({sname}): {n} cells written")
wb.save("AutoEIT_Sample_Audio_for_Transcribing_COMPLETED.xlsx")
print("Saved!")

  038010 (38010-2A): 30 cells written
  038011 (38011-1A): 30 cells written
  038012 (38012-2A): 30 cells written
  038015 (38015-1A): 30 cells written
Saved!


## Step 10: Download

In [10]:
files.download("AutoEIT_Sample_Audio_for_Transcribing_COMPLETED.xlsx")
with open("transcriptions_raw.json","w",encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
files.download("transcriptions_raw.json")
print("Downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded!
